# Football-LLM: QLoRA Fine-tuning on Google Colab

Fine-tune `meta-llama/Llama-3.1-8B-Instruct` to predict FIFA World Cup match results.

**Training data:** 384 samples (2010 + 2014 + 2018 World Cups, named + anonymized variants)  
**Eval data:** 128 samples (2022 World Cup)  
**Method:** QLoRA (4-bit quantization + LoRA r=16)  
**GPU:** Colab free-tier T4 (16 GB VRAM)  

Based on [How to fine-tune open LLMs in 2025](https://www.philschmid.de/fine-tune-llms-in-2025) by Phil Schmid.

## 1. Setup Environment

In [ ]:
# Install dependencies
!pip install -q \
    "torch==2.4.1" \
    "transformers>=4.46.0" \
    "datasets>=3.1.0" \
    "accelerate>=1.1.0" \
    "peft>=0.13.0" \
    "trl>=0.12.0" \
    "bitsandbytes>=0.44.0" \
    "tensorboard" \
    "hf-transfer" \
    "huggingface-hub"

In [ ]:
# Login to HuggingFace Hub (required for Llama model access + pushing weights)
from huggingface_hub import login

# Paste your HuggingFace token here or use Colab secrets
login(token="", add_to_git_credential=True)

In [ ]:
# Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

## 2. Upload Training Data

Upload `train.jsonl` and `eval.jsonl` from `data/training/` to the Colab runtime.
You can either:
- Upload directly via Colab's file browser (left sidebar)
- Mount Google Drive and copy from there
- Clone the repo

In [ ]:
import os

# Option A: Mount Google Drive (upload JSONL files to Drive first)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/football-llm/data/training/*.jsonl /content/

# Option B: Clone the repo (if data is committed)
# !git clone https://github.com/zanwenfu/football-llm.git
# TRAIN_DATA = "football-llm/data/training/train.jsonl"
# EVAL_DATA = "football-llm/data/training/eval.jsonl"

# Option C: Upload via Colab UI — files land in /content/
# from google.colab import files
# uploaded = files.upload()  # select train.jsonl and eval.jsonl

# Set paths (adjust based on upload method)
TRAIN_DATA = "train.jsonl"
EVAL_DATA = "eval.jsonl"

# Verify
for f in [TRAIN_DATA, EVAL_DATA]:
    if os.path.exists(f):
        with open(f) as fh:
            lines = fh.readlines()
        print(f"{f}: {len(lines)} samples")
    else:
        print(f"WARNING: {f} not found! Upload it first.")

## 3. Load Dataset & Inspect

In [ ]:
from datasets import load_dataset

train_dataset = load_dataset("json", data_files=TRAIN_DATA, split="train")
eval_dataset = load_dataset("json", data_files=EVAL_DATA, split="train")

print(f"Train: {len(train_dataset)} samples")
print(f"Eval:  {len(eval_dataset)} samples")
print(f"Features: {list(train_dataset.features.keys())}")
print(f"\nSample messages structure:")
for msg in train_dataset[0]["messages"]:
    print(f"  {msg['role']}: {msg['content'][:100]}...")

## 4. Configure QLoRA Training

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

# ---- Model config ----
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
OUTPUT_DIR = "runs/football-llm-qlora"

# ---- QLoRA quantization ----
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_storage=torch.float16,
)

# ---- LoRA config ----
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    task_type=TaskType.CAUSAL_LM,
)

# ---- Training hyperparameters ----
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_seq_length=2048,
    packing=False,
    # Training
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    # Optimizer
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    max_grad_norm=1.0,
    # Eval
    eval_strategy="epoch",
    # Precision (T4 = fp16)
    fp16=True,
    bf16=False,
    # Logging
    logging_strategy="steps",
    logging_steps=5,
    report_to="tensorboard",
    # Checkpointing
    save_strategy="epoch",
    save_total_limit=3,
    # Hub
    push_to_hub=True,
    hub_strategy="every_save",
    seed=42,
)

print("Configuration ready.")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Total steps: ~{len(train_dataset) * training_args.num_train_epochs // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}")

## 5. Load Model & Tokenizer

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model in 4-bit
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    attn_implementation="sdpa",
    torch_dtype=torch.float16,
    use_cache=False,  # incompatible with gradient checkpointing
    low_cpu_mem_usage=True,
)

print(f"Model loaded: {model.config._name_or_path}")
print(f"Parameters: {model.num_parameters() / 1e9:.2f}B")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 6. Train

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

# Print trainable params (should be ~0.3-0.5% of total)
trainer.model.print_trainable_parameters()

In [ ]:
# Start training
train_result = trainer.train()

# Log metrics
metrics = train_result.metrics
metrics["train_samples"] = len(train_dataset)
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)

## 7. Save Model

In [ ]:
# Restore cache for inference
trainer.model.config.use_cache = True

# Save adapter weights + tokenizer
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Push to hub
trainer.push_to_hub()

print(f"\nAdapter saved to: {OUTPUT_DIR}")
print(f"Adapter size: {sum(f.stat().st_size for f in __import__('pathlib').Path(OUTPUT_DIR).rglob('*') if f.is_file()) / 1e6:.1f} MB")

## 8. Merge Adapter Weights (Optional)

Merge the LoRA adapter into the base model for standalone deployment.
Only needed if you want to serve the model without PEFT / vLLM's LoRA support.

In [ ]:
from peft import AutoPeftModelForCausalLM

# Free GPU memory from training
del model, trainer
torch.cuda.empty_cache()

# Load and merge
merged_model = AutoPeftModelForCausalLM.from_pretrained(
    OUTPUT_DIR,
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16,
)
merged_model = merged_model.merge_and_unload()

# Save merged model
MERGED_DIR = "runs/football-llm-merged"
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True, max_shard_size="3GB")
tokenizer.save_pretrained(MERGED_DIR)

print(f"Merged model saved to: {MERGED_DIR}")

## 9. Quick Inference Test

Verify the fine-tuned model generates sensible predictions.

In [ ]:
from peft import PeftModel

# Free memory
try:
    del merged_model
except NameError:
    pass
torch.cuda.empty_cache()

# Reload base model + adapter for inference
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    attn_implementation="sdpa",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)
ft_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
ft_model.eval()

# Take a sample from eval set
import json
sample = eval_dataset[0]
messages = sample["messages"][:2]  # system + user only

print("=" * 60)
print("INPUT (user message preview):")
print(messages[1]["content"][:500])
print("=" * 60)

# Generate
input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = ft_model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("\nMODEL PREDICTION:")
print(response)
print("\nGROUND TRUTH:")
print(sample["messages"][2]["content"][:300])

## 10. TensorBoard

View training loss curves.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/football-llm-qlora